# Credit Default Risk Scorecard
## Notebook 1: Exploratory Data Analysis
**Author:** Simpson Gundlapally
**Dataset:** Give Me Some Credit — Kaggle
**Objective:** Understand the dataset, identify data quality issues, and surface initial patterns in credit default behaviour.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_r')

print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/cs-training.csv', index_col=0)

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
df.head()

## 2. Column Overview

In [ ]:
# Data dictionary mapping
col_descriptions = {
    'SeriousDlqin2yrs': 'TARGET: Person 90+ days past due (1=Yes, 0=No)',
    'RevolvingUtilizationOfUnsecuredLines': 'Credit card balance / credit limit ratio',
    'age': 'Age of borrower in years',
    'NumberOfTime30-59DaysPastDueNotWorse': 'Times 30-59 days past due (last 2 years)',
    'DebtRatio': 'Monthly debt / monthly gross income',
    'MonthlyIncome': 'Monthly income (USD)',
    'NumberOfOpenCreditLinesAndLoans': 'Open loans + lines of credit',
    'NumberOfTimes90DaysLate': 'Times 90+ days past due',
    'NumberRealEstateLoansOrLines': 'Mortgage and real estate loans',
    'NumberOfTime60-89DaysPastDueNotWorse': 'Times 60-89 days past due (last 2 years)',
    'NumberOfDependents': 'Number of dependents (excl. self)'
}

for col, desc in col_descriptions.items():
    print(f'{col:45} | {desc}')

## 3. Data Types & Missing Values

In [ ]:
# Missing values analysis
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
    'Data Type': df.dtypes
})
missing[missing['Missing Count'] > 0]

## 4. Target Variable Analysis

In [ ]:
target_counts = df['SeriousDlqin2yrs'].value_counts()
default_rate = df['SeriousDlqin2yrs'].mean()

print(f'Total borrowers: {len(df):,}')
print(f'Defaulted (1): {target_counts[1]:,} ({target_counts[1]/len(df)*100:.1f}%)')
print(f'Non-defaulted (0): {target_counts[0]:,} ({target_counts[0]/len(df)*100:.1f}%)')
print(f'Overall default rate: {default_rate:.2%}')
print(f'\nClass imbalance ratio: 1:{target_counts[0]//target_counts[1]}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Default (0)', 'Default (1)'], target_counts.values,
            color=['#0064A4', '#E84040'])
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=['#0064A4', '#E84040'],
            startangle=90)
axes[1].set_title('Default Rate Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../dashboard/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. Default Rate by Key Variables

In [ ]:
# Age bands
df['AgeBand'] = pd.cut(df['age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['Under 25', '25-35', '35-45', '45-55', '55-65', 'Over 65'])

age_default = df.groupby('AgeBand')['SeriousDlqin2yrs'].agg(['mean', 'count'])
age_default.columns = ['Default Rate', 'Count']
age_default['Default Rate'] = age_default['Default Rate'] * 100

print('Default Rate by Age Band:')
print(age_default.round(2))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(age_default.index, age_default['Default Rate'],
              color=['#E84040' if x > age_default['Default Rate'].mean()
                     else '#0064A4' for x in age_default['Default Rate']])
ax.axhline(y=age_default['Default Rate'].mean(), color='orange',
           linestyle='--', linewidth=2, label=f'Average: {age_default["Default Rate"].mean():.1f}%')
ax.set_title('Default Rate by Age Band', fontsize=13, fontweight='bold')
ax.set_ylabel('Default Rate (%)')
ax.legend()
for bar, val in zip(bars, age_default['Default Rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('../dashboard/02_default_by_age.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Utilisation bands
df['UtilBand'] = pd.cut(df['RevolvingUtilizationOfUnsecuredLines'],
    bins=[-0.001, 0.3, 0.6, 0.9, float('inf')],
    labels=['Low (0-30%)', 'Medium (30-60%)', 'High (60-90%)', 'Very High (90%+)'])

util_default = df.groupby('UtilBand')['SeriousDlqin2yrs'].agg(['mean', 'count'])
util_default.columns = ['Default Rate', 'Count']
util_default['Default Rate'] = util_default['Default Rate'] * 100

print('Default Rate by Utilisation Band:')
print(util_default.round(2))

# Key finding
low = util_default.loc['Low (0-30%)', 'Default Rate']
high = util_default.loc['Very High (90%+)', 'Default Rate']
print(f'\nKEY FINDING: Very High utilisation borrowers default at {high/low:.1f}x the rate of Low utilisation borrowers')

## 6. Correlation with Default

In [ ]:
# Correlation of all features with target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
correlations = df[numeric_cols].corr()['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs')
correlations = correlations.sort_values(ascending=False)

print('Correlation with Default (SeriousDlqin2yrs):')
for col, corr in correlations.items():
    bar = '█' * int(abs(corr) * 50)
    direction = '+' if corr > 0 else '-'
    print(f'{col:50} {direction}{bar} {corr:.4f}')

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E84040' if c > 0 else '#0064A4' for c in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.set_title('Feature Correlation with Default', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../dashboard/03_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. EDA Summary

**Key Findings:**
- Dataset contains 150,000 borrowers with a **~6.7% default rate** (class imbalance 1:14)
- **Two columns have missing values:** MonthlyIncome and NumberOfDependents
- **Younger borrowers (Under 25, 25-35) have the highest default rates**
- **Very High utilisation borrowers default at significantly higher rates than Low utilisation**
- **Most correlated with default:** NumberOfTimes90DaysLate, NumberOfTime30-59DaysPastDueNotWorse, RevolvingUtilizationOfUnsecuredLines

**Next Steps:** Feature Engineering → handle nulls, create new features, prepare for modelling